# Code mẫu — Bài toán mở

Có điểm số từ một model chưa có nghĩa là có lời giải hợp lệ. Còn phải chọn câu hỏi cho khéo, giữ đúng các ràng buộc của đề, và kiểm xem các giả định của mình còn đúng khi gặp tình huống mới hay không. Các bài nhỏ dưới đây dùng điểm và tọa độ tự đặt, không nạp từ điển hay model pretrained của cuộc thi nào.

## Ví dụ 1: Nên hỏi câu nào để thu hẹp nhiều nhất?

Có tám đối tượng nằm ở các tọa độ từ 0 tới 7. Một đối tượng bí mật được giấu đi, gọi là hidden. Ta đang giữ một ứng viên tốt nhất, gọi là champion, ở tọa độ 0. Mỗi lượt, ta đưa ra một đề xuất, gọi là proposal. Bộ chấm, gọi là judge, cho biết champion gần hidden hơn, hai bên gần bằng nhau, hay proposal gần hơn. Chưa biết hidden ở đâu, nên ta coi cả tám vị trí đều có khả năng như nhau. Ta so hai đề xuất, 1 và 6, theo số ứng viên dự kiến còn lại sau câu trả lời.

In [ ]:
from collections import Counter
import numpy as np

points = np.arange(8, dtype=float)
def verdict(a, b, hidden):
    left, right = abs(points[a] - points[hidden]), abs(points[b] - points[hidden])
    return 0 if left == right else (-1 if left < right else 1)

for proposal in [1, 6]:
    partitions = {response: [hidden for hidden in range(8)
                             if verdict(0, proposal, hidden) == response]
                  for response in [-1, 0, 1]}
    sizes = [len(group) for group in partitions.values()]
    expected = sum(size ** 2 for size in sizes) / 8
    # Duyệt từng hidden một, để kiểm công thức tính theo kích thước nhóm bằng một cách độc lập.
    enumerated = np.mean([len(partitions[verdict(0, proposal, hidden)])
                          for hidden in range(8)])
    assert expected == enumerated
    print("proposal:", proposal, "nhóm:", partitions)
    print("số còn lại kỳ vọng:", f"{expected:.2f}")

proposal: 1 nhóm: {-1: [0], 0: [], 1: [1, 2, 3, 4, 5, 6, 7]}
số còn lại kỳ vọng: 6.25
proposal: 6 nhóm: {-1: [0, 1, 2], 0: [3], 1: [4, 5, 6, 7]}
số còn lại kỳ vọng: 3.25


Mỗi câu trả lời chia tám khả năng thành ba nhóm. Nếu nhóm có k phần tử, mà hidden rơi vào nhóm đó với xác suất k/8, thì số ứng viên còn lại trung bình là tổng $k^2/8$.

Đề xuất 1 chia thành ba nhóm cỡ 1, 0, 7. Gần như chắc chắn câu trả lời sẽ là “proposal gần hơn”, và còn tới 7 ứng viên. Trung bình còn 6.25. Đề xuất 6 chia thành 3, 1, 4, và trung bình chỉ còn 3.25. Đề xuất nằm sát champion lại là câu hỏi ít thông tin nhất. Cách tính này giả định mọi vị trí có khả năng như nhau, và judge trả lời đúng theo mô hình khoảng cách của ta. Nếu một trong hai giả định sai, tiêu chí này phải xem lại.

## Ví dụ 2: Ghép các đoạn, mỗi đoạn dùng đúng một lần

Có bốn đoạn A, B, C, D. Một model nhỏ chấm điểm cho từng cách nối hai đoạn liền nhau. A bắt buộc đứng đầu, D bắt buộc đứng cuối. Nếu cứ chọn cặp có điểm cao nhất một cách riêng lẻ, ta có thể dùng B hai lần và bỏ quên C. Ta liệt kê mọi thứ tự hợp lệ, rồi chấm tổng điểm của ba lần nối.

In [ ]:
from itertools import permutations

pieces = ("A", "B", "C", "D")
score = {("A", "B"): 9, ("A", "C"): 7,
         ("B", "C"): 1, ("B", "D"): 8,
         ("C", "B"): 8, ("C", "D"): 2}
answers = []
for order in permutations(pieces):
    if order[0] != "A" or order[-1] != "D":
        continue
    value = sum(score[left, right] for left, right in zip(order, order[1:]))
    assert len(set(order)) == len(pieces)
    answers.append((value, order))
for value, order in sorted(answers):
    print("thứ tự:", order, "tổng điểm:", value)
best = max(answers)
assert best == (23, ("A", "C", "B", "D"))
print("cặp đầu cao nhất:", max((value, pair) for pair, value in score.items() if pair[0] == "A"))
print("lời giải toàn cục:", best)

thứ tự: ('A', 'B', 'C', 'D') tổng điểm: 12
thứ tự: ('A', 'C', 'B', 'D') tổng điểm: 23
cặp đầu cao nhất: (9, ('A', 'B'))
lời giải toàn cục: (23, ('A', 'C', 'B', 'D'))


Chọn tham lam A sang B ở bước đầu, vì điểm 9 là cao nhất. Nhưng sau đó chỉ còn cách đi B, C, D, với tổng $9+1+2=12$. Thứ tự A, C, B, D bắt đầu thấp hơn, chỉ 7, nhưng tổng đạt $7+8+8=23$. Một bước đầu đẹp có thể dẫn vào ngõ cụt.

Ở đây chỉ có bốn đoạn nên duyệt hết được. Với nhiều đoạn hơn, cần một thuật toán tìm kiếm khôn hơn, nhưng ràng buộc mỗi đoạn dùng đúng một lần vẫn phải giữ. Điểm 23 là tối ưu theo bảng điểm này. Nó chưa chứng minh câu chuyện ghép lại có nghĩa, vì bảng điểm của model cũng có thể sai.

## Ví dụ 3: Giữ nguyên người chơi, đổi cách judge nhìn thế giới

Người chơi, gọi là player, chỉ biết các tọa độ công khai từ 0 tới 7. Player được hỏi tối đa ba lần, và không được hỏi lại một đề xuất. Mỗi lượt, player chọn câu hỏi theo tiêu chí ở ví dụ 1, rồi loại hẳn những ứng viên không khớp với câu trả lời. Ta thử player trên cả tám vị trí hidden, với judge dùng đúng tọa độ công khai. Sau đó giữ nguyên player và số lượt, nhưng cho judge dùng một bản tọa độ riêng, trong đó vị trí của ID 4 và ID 5 bị đổi cho nhau.

In [ ]:
import numpy as np

public = np.arange(8, dtype=float)
def predict_response(points, a, b, hidden):
    left, right = abs(points[a] - points[hidden]), abs(points[b] - points[hidden])
    return 0 if left == right else (-1 if left < right else 1)

def make_judge(private_points, hidden):
    asked = set()
    def ask(champion, proposal):
        assert 0 <= proposal < len(private_points) and proposal not in asked
        asked.add(proposal)
        # Chỉ judge biết hidden. Player chỉ nhận về: có trúng không, và một câu trả lời.
        return proposal == hidden, predict_response(private_points, champion, proposal, hidden)
    return ask

def player(ask, budget=3):
    possible, asked, champion = list(range(len(public))), set(), 0
    for turn in range(1, budget + 1):
        available = [item for item in possible if item not in asked]
        if not available:
            return False, turn - 1
        def expected(proposal):
            counts = [sum(predict_response(public, champion, proposal, h) == response
                          for h in possible) for response in [-1, 0, 1]]
            return sum(count ** 2 for count in counts) / len(possible)
        proposal = min(available, key=lambda q: (expected(q), q))
        asked.add(proposal)
        hit, response = ask(champion, proposal)
        if hit:
            return True, turn
        possible = [h for h in possible if h != proposal and
                    predict_response(public, champion, proposal, h) == response]
        if response == 1:
            champion = proposal
    return False, budget

shifted = public.copy()
shifted[[4, 5]] = shifted[[5, 4]]
for name, private in [("cùng miền", public), ("đổi 4 và 5", shifted)]:
    results = [player(make_judge(private, hidden)) for hidden in range(8)]
    solved = sum(success for success, turns in results)
    print(name, "giải được:", solved, "/ 8", "từng game:", results)

response = predict_response(shifted, 4, 5, 6)
kept = [h for h in range(8) if predict_response(public, 4, 5, h) == response]
penalty = np.maximum(0, abs(public - public[4]) - abs(public - public[5]))
print("phản hồi judge cho (4,5):", response, "lọc giữ:", kept)
print("hidden 6 còn sau lọc:", 6 in kept, "penalty mềm:", float(penalty[6]))
assert 6 not in kept and np.isfinite(penalty[6])

cùng miền giải được: 8 / 8 từng game: [(True, 3), (True, 3), (True, 2), (True, 2), (True, 2), (True, 3), (True, 1), (True, 3)]
đổi 4 và 5 giải được: 7 / 8 từng game: [(True, 3), (True, 3), (True, 2), (True, 2), (True, 2), (False, 2), (True, 1), (True, 3)]
phản hồi judge cho (4,5): -1 lọc giữ: [0, 1, 2, 3, 4]
hidden 6 còn sau lọc: False penalty mềm: 1.0


Cách dựng code giữ cho phép thử công bằng. Hàm `player` không hề nhận hidden làm tham số. Nó chỉ được hỏi qua hàm `ask`, và mọi game đều bị giới hạn ba lượt như nhau. Các game thua vẫn được in ra, không bị lọc bỏ.

Khi judge nhìn đúng như player, player giải được cả 8 game. Đổi chỗ hai tọa độ ở phía judge, player giữ nguyên mà chỉ còn giải được 7. Ở game hidden 5, sau hai lượt, bước loại cứng đã xóa mất đáp án thật, và không còn ứng viên nào để hỏi.

Ba dòng cuối minh họa cơ chế đó bằng một phép thử riêng, với hidden 6 và cặp `(4, 5)`. Đây không phải diễn biến của game hidden 6, vì game ấy đã trúng ngay lượt đầu. Theo tọa độ riêng của judge, câu trả lời là −1. Theo tọa độ công khai mà player dùng để lọc, hidden 6 lẽ ra phải cho câu trả lời khác, nên bị loại hẳn. Nếu thay việc loại cứng bằng một mức phạt mềm, hidden 6 chỉ bị trừ 1 điểm và vẫn còn trong danh sách. Giữ lại ứng viên như vậy chỉ giúp không mất đáp án vĩnh viễn. Nó chưa bảo đảm xếp hạng đúng, hay giải được trong số lượt cho phép.